# Swiggy Annual Report QA — Pipeline Evaluation

This notebook walks through each step of the RAG pipeline, showing intermediate outputs
and statistics at every stage. Use this to understand how the system works internally.

**Pipeline:** PDF → Text Extraction → Chunking → Embeddings → FAISS Index → Retrieval → Gemini Answer

## Step 0: Setup & Imports

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

from pdf_loader import load_pdf
from text_chunker import make_chunks, clean_ocr_text
from embeddings import load_model, get_embeddings
from vector_store import create_index, save_to_disk, load_from_disk, find_similar
from query_engine import init_gemini, ask, make_prompt

import numpy as np
print('All modules imported successfully.')

## Step 1: PDF Text Extraction

We use **PyMuPDF** for text-based pages and **Tesseract OCR** for scanned image pages.
The Swiggy report has 170 pages — roughly 30 text pages and 140 scanned pages.

In [ ]:
PDF_PATH = os.path.join('data', 'Swiggy Annual-Report-FY-2023-24.pdf')

# This takes a few minutes due to OCR on scanned pages
pages = load_pdf(PDF_PATH)

print(f'\nTotal pages extracted: {len(pages)}')
print(f'Page number range: {pages[0]["page_num"]} to {pages[-1]["page_num"]}')

In [ ]:
# Preview text from a few pages
for page in pages[:3]:
    print(f'--- Page {page["page_num"]} (first 300 chars) ---')
    print(page['text'][:300])
    print()

## Step 2: Text Chunking

Raw text is cleaned (OCR noise removed) and split into **overlapping word-based chunks**.
- Chunk size: 400 words
- Overlap: 80 words (ensures important sentences aren't cut at boundaries)

In [ ]:
chunks = make_chunks(pages, chunk_sz=400, overlap=80)

print(f'Total chunks created: {len(chunks)}')
print(f'Average chunk length: {np.mean([len(c["text"].split()) for c in chunks]):.0f} words')
print(f'Min chunk length: {min(len(c["text"].split()) for c in chunks)} words')
print(f'Max chunk length: {max(len(c["text"].split()) for c in chunks)} words')

In [ ]:
# Show a sample chunk
sample = chunks[10]
print(f'Chunk ID: {sample["chunk_id"]}')
print(f'Source page: {sample["page_num"]}')
print(f'Word count: {len(sample["text"].split())}')
print(f'\nText preview:\n{sample["text"][:500]}')

In [ ]:
# Distribution of chunks per page
from collections import Counter
page_counts = Counter(c['page_num'] for c in chunks)
print(f'Pages with chunks: {len(page_counts)}')
print(f'Max chunks from a single page: {max(page_counts.values())}')
print(f'Pages with most chunks: {page_counts.most_common(5)}')

## Step 3: Embedding Generation

Using **all-MiniLM-L6-v2** (384-dimensional vectors). This model is:
- Lightweight (~80MB)
- Fast to encode
- Effective for semantic similarity

In [ ]:
embedding_model = load_model()
embeddings = get_embeddings(chunks, embedding_model)

print(f'\nEmbedding matrix shape: {embeddings.shape}')
print(f'Embedding dimension: {embeddings.shape[1]}')
print(f'Data type: {embeddings.dtype}')
print(f'Memory size: {embeddings.nbytes / 1024:.1f} KB')

In [ ]:
# Show sample embedding values
print('First embedding (first 20 values):')
print(embeddings[0][:20])

## Step 4: FAISS Vector Index

We build a **FAISS IndexFlatL2** for exact nearest-neighbor search using L2 (Euclidean) distance.

In [ ]:
index = create_index(embeddings)

print(f'\nIndex type: IndexFlatL2')
print(f'Total vectors: {index.ntotal}')
print(f'Dimension: {index.d}')
print(f'Is trained: {index.is_trained}')

## Step 5: Semantic Search (Retrieval)

Given a user question, we:
1. Encode the question with the same embedding model
2. Search FAISS for the top-K most similar chunks
3. Return chunks with relevance scores

In [ ]:
question = 'What was Swiggy\'s total revenue in FY 2023-24?'
query_embedding = embedding_model.encode([question])
query_embedding = np.array(query_embedding, dtype='float32')

print(f'Query: "{question}"')
print(f'Query embedding shape: {query_embedding.shape}')

results = find_similar(query_embedding, index, chunks, top_k=5)

print(f'\nTop 5 retrieved chunks:')
for i, r in enumerate(results, 1):
    print(f'\n[{i}] Page {r["page_num"]} | Score: {r["score"]:.4f}')
    print(f'    {r["text"][:200]}...')

## Step 6: Prompt Construction

The retrieved chunks are formatted into a prompt with **anti-hallucination instructions**.
The system prompt tells Gemini to:
- Answer only from the provided context
- Say "I don't know" if the answer isn't found
- Cite page numbers

In [ ]:
prompt = make_prompt(question, results)

print(f'Prompt length: {len(prompt)} characters')
print(f'Prompt word count: {len(prompt.split())} words')
print(f'\n--- Full Prompt (first 1500 chars) ---')
print(prompt[:1500])
print('...')

## Step 7: Full QA Pipeline — End-to-End Examples

Now let's run the complete pipeline: Question → Retrieve → Generate Answer.

**Note:** You need a valid Gemini API key to run this step.

In [ ]:
# load pre-built index instead of rebuilding
index, chunks = load_from_disk('vector_db')

gemini = init_gemini()  # uses built-in API key

In [ ]:
# Test questions
test_questions = [
    'What was Swiggy\'s total revenue in FY 2023-24?',
    'Who is the CEO of Swiggy?',
    'What was Swiggy\'s total loss in FY24?',
    'What was Swiggy\'s revenue in FY 2025?',  # Anti-hallucination test
]

for q in test_questions:
    result = ask(q, gemini, index, chunks, embedding_model)
    print(f'Q: {q}')
    print(f'A: {result["answer"]}')
    print(f'Sources: {result["sources"]}')
    print('-' * 60)
    print()

## Summary

| Stage | Description | Output |
|---|---|---|
| PDF Extraction | PyMuPDF + Tesseract OCR | 170 pages of text |
| Chunking | 400-word overlapping chunks | ~319 chunks |
| Embeddings | all-MiniLM-L6-v2 (384-dim) | (319, 384) matrix |
| Vector Index | FAISS IndexFlatL2 | 319 vectors indexed |
| Retrieval | Top-5 semantic search | Relevant chunks + scores |
| Generation | Gemini 2.5 Flash | Grounded answer + citations |

**Anti-hallucination** is verified by the FY 2025 question — the model correctly responds
"I don't know based on the document." when the answer is not in the report.